In [1]:
!pip install -q --upgrade diffusers accelerate transformers imageio imageio-ffmpeg Pillow
print('✅ Done!-now Runtime> Restart Session')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 37.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
✅ Done!-now Runtime> Restart Session


In [1]:
import torch, os, time, numpy as np
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video
from IPython.display import Video, display

CONFIG = {
    'model_id': 'damo-vilab/text-to-video-ms-1.7b',
    'num_frames': 16,
    'fps': 8,
    'width': 256,
    'height': 256,
    'num_inference_steps': 40,
    'guidance_scale': 5.0,
    'seed': 42,
    'negative_prompt': 'blurry, low quality, distorted, flickering, watermark',
    'output_dir': 'outputs',
}
print('✅ Ready')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Ready


In [2]:
pipe = DiffusionPipeline.from_pretrained(
    CONFIG['model_id'], torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
print('✅ Model ready!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


✅ Model ready!


In [3]:
STYLE_PRESETS = {
    'cinematic': 'cinematic lighting, professional camera, smooth motion, high quality',
    'nature':    'golden hour lighting, natural colors, peaceful atmosphere',
    'dramatic':  'dramatic lighting, high contrast, intense atmosphere, epic scale',
    'minimal':   'clean composition, soft lighting, minimalist style',
}

def enhance_prompt(raw, style='cinematic'):
    if len(raw.split()) > 15:
        return raw
    return f'{raw}, {STYLE_PRESETS.get(style, STYLE_PRESETS["cinematic"])}'

def generate_video(prompt, output_filename=None, cfg_scale=None,
                   steps=None, seed=None, style='cinematic'):
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    cfg_scale = cfg_scale or CONFIG['guidance_scale']
    steps     = steps     or CONFIG['num_inference_steps']
    seed      = seed      or CONFIG['seed']
    prompt    = enhance_prompt(prompt, style)
    if output_filename is None:
        safe = prompt[:25].replace(' ', '_').replace(',', '')
        output_filename = safe + '.mp4'
    output_path = os.path.join(CONFIG['output_dir'], output_filename)
    print(f'Prompt : {prompt}')
    print(f'CFG: {cfg_scale} | Steps: {steps} | Seed: {seed}')
    start = time.time()
    result = pipe(
        prompt=prompt,
        negative_prompt=CONFIG['negative_prompt'],
        num_frames=CONFIG['num_frames'],
        num_inference_steps=steps,
        guidance_scale=cfg_scale,
        width=CONFIG['width'],
        height=CONFIG['height'],
        generator=torch.manual_seed(seed),
    )
    frames = result.frames[0]
    if hasattr(frames[0], 'convert'):
        frames = [np.array(f) for f in frames]
    export_to_video(frames, output_path, fps=CONFIG['fps'])
    print(f'✅ Done in {time.time()-start:.1f}s → {output_path}')
    return output_path

print('✅ Functions ready!')

✅ Functions ready!


In [4]:
prompts = [
    ('Ocean waves crashing on a beach at sunset', 'video_1.mp4', 'nature'),
    ('A fire burning brightly at night', 'video_2.mp4', 'dramatic'),
    ('A rocket launching into space', 'video_3.mp4', 'cinematic'),
]

for prompt, filename, style in prompts:
    print(f'\n--- {prompt} ---')
    path = generate_video(prompt, output_filename=filename, style=style)
    display(Video(path, embed=True, width=400))


--- Ocean waves crashing on a beach at sunset ---
Prompt : Ocean waves crashing on a beach at sunset, golden hour lighting, natural colors, peaceful atmosphere
CFG: 5.0 | Steps: 40 | Seed: 42


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Done in 34.3s → outputs/video_1.mp4



--- A fire burning brightly at night ---
Prompt : A fire burning brightly at night, dramatic lighting, high contrast, intense atmosphere, epic scale
CFG: 5.0 | Steps: 40 | Seed: 42


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Done in 32.5s → outputs/video_2.mp4



--- A rocket launching into space ---
Prompt : A rocket launching into space, cinematic lighting, professional camera, smooth motion, high quality
CFG: 5.0 | Steps: 40 | Seed: 42


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Done in 33.2s → outputs/video_3.mp4


In [ ]:
from google.colab import files
for i in range(1, 4):
    files.download(f'outputs/video_{i}.mp4')

In [5]:
import gradio as gr
print(f'Gradio version: {gr.__version__}')

Gradio version: 5.50.0


In [6]:
# Frame interpolation using OpenCV
import cv2
import numpy as np

def interpolate_frames(frames, multiplier=2):
    """
    Takes a list of frames and inserts blended frames between each pair.
    multiplier=2 means double the frames (16 → 32)
    multiplier=3 means triple the frames (16 → 48)
    """
    if multiplier == 1:
        return frames

    interpolated = []

    for i in range(len(frames) - 1):
        frame_current = np.array(frames[i]).astype(np.float32)
        frame_next    = np.array(frames[i + 1]).astype(np.float32)

        # Always add the current frame
        interpolated.append(frames[i])

        # Add (multiplier-1) blended frames between current and next
        for j in range(1, multiplier):
            alpha = j / multiplier   # blending weight 0→1
            # Linear interpolation: blend = current*(1-alpha) + next*alpha
            blended = cv2.addWeighted(
                frame_current, 1 - alpha,
                frame_next,    alpha,
                0
            ).astype(np.uint8)
            interpolated.append(blended)

    # Add the last frame
    interpolated.append(frames[-1])

    return interpolated

In [7]:
def generate(prompt, style, cfg_scale, steps, seed, interpolation):
    if not prompt.strip():
        return None, 'Please enter a prompt!', format_history()

    enhanced = enhance_prompt(prompt, style)
    os.makedirs('outputs', exist_ok=True)

    safe     = prompt[:20].replace(' ', '_').replace(',', '')
    filename = f'outputs/{safe}_{int(time.time())}.mp4'

    print(f'Generating: {enhanced}')
    start = time.time()

    result = pipe(
        prompt=enhanced,
        negative_prompt='blurry, low quality, distorted, flickering, watermark',
        num_frames=16,
        num_inference_steps=int(steps),
        guidance_scale=cfg_scale,
        width=256,
        height=256,
        generator=torch.manual_seed(int(seed)),
    )

    frames = result.frames[0]
    if hasattr(frames[0], 'convert'):
        frames = [np.array(f) for f in frames]

    # Apply frame interpolation if selected
    multiplier = {'None': 1, '2x Smoother': 2, '3x Smoother': 3}[interpolation]
    if multiplier > 1:
        frames = interpolate_frames(frames, multiplier)
        print(f'Interpolated: {16} → {len(frames)} frames')

    # FPS also increases with interpolation so video length stays same
    output_fps = 8 * multiplier
    export_to_video(frames, filename, fps=output_fps)

    elapsed = time.time() - start
    save_to_history(prompt, style, enhanced, filename, elapsed)

    status = (
        f'✅ Done in {elapsed:.1f}s\n'
        f'Frames: {len(frames)} | FPS: {output_fps} | Style: {style}'
    )
    return filename, status, format_history()

print('✅ generate() with interpolation ready!')

✅ generate() with interpolation ready!


In [8]:
import gradio as gr
import torch
import cv2
import os
import time
import json
import numpy as np
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video
# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
CONFIG = {
    'model_id': 'damo-vilab/text-to-video-ms-1.7b',
    'num_frames': 16,
    'fps': 8,
    'width': 256,
    'height': 256,
    'num_inference_steps': 25,
    'guidance_scale': 7.5,
    'seed': 42,
    'negative_prompt': 'blurry, low quality, distorted, flickering, watermark',
    'output_dir': 'outputs',
    'history_file': 'history.json',
}

# ─────────────────────────────────────────
# STYLE PRESETS
# ─────────────────────────────────────────
STYLE_PRESETS = {
    'Cinematic': 'cinematic lighting, professional camera, smooth motion, high quality',
    'Nature':    'golden hour lighting, natural colors, peaceful atmosphere',
    'Dramatic':  'dramatic lighting, high contrast, intense atmosphere, epic scale',
    'Minimal':   'clean composition, soft lighting, minimalist style',
    'Anime':     'anime style, vibrant colors, smooth animation, Studio Ghibli',
}

def enhance_prompt(raw, style):
    if len(raw.split()) > 15:
        return raw
    style_tags = STYLE_PRESETS.get(style, '')
    return f'{raw}, {style_tags}' if style_tags else raw

# ─────────────────────────────────────────
# FRAME INTERPOLATION
# ─────────────────────────────────────────
def interpolate_frames(frames, multiplier=2):
    if multiplier == 1:
        return frames
    interpolated = []
    for i in range(len(frames) - 1):
        frame_current = np.array(frames[i]).astype(np.float32)
        frame_next    = np.array(frames[i + 1]).astype(np.float32)
        interpolated.append(frames[i])
        for j in range(1, multiplier):
            alpha   = j / multiplier
            blended = cv2.addWeighted(
                frame_current, 1 - alpha,
                frame_next,    alpha, 0
            ).astype(np.uint8)
            interpolated.append(blended)
    interpolated.append(frames[-1])
    return interpolated

# ─────────────────────────────────────────
# TEMPORAL BLENDING — fixes motion flicker
# ─────────────────────────────────────────
def smooth_frames(frames):
    smoothed = []
    frames_f = [np.array(f).astype(np.float32) for f in frames]
    for i in range(len(frames_f)):
        if i == 0:
            blended = frames_f[i] * 0.7 + frames_f[i+1] * 0.3
        elif i == len(frames_f) - 1:
            blended = frames_f[i] * 0.7 + frames_f[i-1] * 0.3
        else:
            blended = (
                frames_f[i-1] * 0.15 +
                frames_f[i]   * 0.70 +
                frames_f[i+1] * 0.15
            )
        blended = blended.astype(np.uint8)
        blended = cv2.GaussianBlur(blended, (3, 3), 0.5)
        smoothed.append(blended)
    return smoothed

# ─────────────────────────────────────────
# HISTOGRAM MATCHING — fixes color flicker
# ─────────────────────────────────────────
def match_histograms(frames):
    reference = np.array(frames[0]).astype(np.uint8)
    matched   = [reference]
    for frame in frames[1:]:
        frame         = np.array(frame).astype(np.uint8)
        matched_frame = np.zeros_like(frame)
        for c in range(3):
            ref_hist = cv2.calcHist([reference], [c], None, [256], [0, 256])
            src_hist = cv2.calcHist([frame],     [c], None, [256], [0, 256])
            ref_cdf  = ref_hist.cumsum() / ref_hist.sum()
            src_cdf  = src_hist.cumsum() / src_hist.sum()
            lookup   = np.zeros(256, dtype=np.uint8)
            j = 0
            for k in range(256):
                while j < 255 and src_cdf[j] < ref_cdf[k]:
                    j += 1
                lookup[k] = j
            matched_frame[:, :, c] = cv2.LUT(frame[:, :, c], lookup)
        matched.append(matched_frame)
    return matched

# ─────────────────────────────────────────
# HISTORY
# ─────────────────────────────────────────
def load_history():
    if os.path.exists(CONFIG['history_file']):
        with open(CONFIG['history_file'], 'r') as f:
            return json.load(f)
    return []

def save_to_history(prompt, style, enhanced, path, duration):
    history = load_history()
    entry   = {
        'prompt':   prompt,
        'style':    style,
        'enhanced': enhanced,
        'path':     path,
        'duration': f'{duration:.1f}s',
        'time':     time.strftime('%Y-%m-%d %H:%M'),
    }
    history.insert(0, entry)
    history = history[:10]
    with open(CONFIG['history_file'], 'w') as f:
        json.dump(history, f, indent=2)
    return history

def format_history(history):
    if not history:
        return 'No generations yet.'
    lines = []
    for i, h in enumerate(history):
        lines.append(
            f"#{i+1} [{h['time']}]\n"
            f"  Prompt : {h['prompt']}\n"
            f"  Style  : {h['style']}\n"
            f"  Time   : {h['duration']}\n"
        )
    return '\n'.join(lines)

# ─────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────
pipe = None

def load_model():
    global pipe
    if pipe is None:
        print('Loading model...')
        pipe = DiffusionPipeline.from_pretrained(
            CONFIG['model_id'],
            torch_dtype=torch.float16,
        )
        pipe.enable_model_cpu_offload()
        print('✅ Model ready!')

# ─────────────────────────────────────────
# GENERATE
# ─────────────────────────────────────────
def generate_video(prompt, style, cfg_scale, steps, seed, interpolation):
    if not prompt.strip():
        return None, 'Please enter a prompt!', format_history(load_history())

    load_model()

    enhanced    = enhance_prompt(prompt, style)
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    safe        = prompt[:20].replace(' ', '_').replace(',', '')
    filename    = f'{safe}_{int(time.time())}.mp4'
    output_path = os.path.join(CONFIG['output_dir'], filename)

    print(f'Prompt : {enhanced}')
    print(f'Style  : {style}')
    print(f'CFG    : {cfg_scale} | Steps: {steps} | Seed: {seed}')

    start  = time.time()
    result = pipe(
        prompt=enhanced,
        negative_prompt=CONFIG['negative_prompt'],
        num_frames=CONFIG['num_frames'],
        num_inference_steps=int(steps),
        guidance_scale=cfg_scale,
        width=CONFIG['width'],
        height=CONFIG['height'],
        generator=torch.manual_seed(int(seed)),
    )

    # Convert frames correctly
    frames = result.frames[0]
    if hasattr(frames[0], 'convert'):
        frames = [np.array(f.convert('RGB')) for f in frames]
    else:
        frames = [np.array(f) for f in frames]

    # Ensure 0-255 range
    frames = [
        (f * 255).astype(np.uint8) if f.max() <= 1.0 else f.astype(np.uint8)
        for f in frames
    ]

    # Step 1 — interpolation
    multiplier = {'None': 1, '2x Smoother': 2, '3x Smoother': 3}[interpolation]
    if multiplier > 1:
        frames = interpolate_frames(frames, multiplier)
        print(f'Frames : 16 → {len(frames)}')

    # Step 2 — histogram matching
    frames = match_histograms(frames)

    # Step 3 — temporal blending
    frames = smooth_frames(frames)

    # Step 4 — export
    output_fps = CONFIG['fps'] * multiplier
    export_to_video(frames, output_path, fps=output_fps)

    elapsed = time.time() - start
    history = save_to_history(prompt, style, enhanced, output_path, elapsed)

    status = (
        f'✅ Done in {elapsed:.1f}s\n'
        f'Frames: {len(frames)} | FPS: {output_fps} | Style: {style}'
    )
    return output_path, status, format_history(history)

# ─────────────────────────────────────────
# GRADIO UI
# ─────────────────────────────────────────
with gr.Blocks(title='Text-to-Video AI', theme=gr.themes.Monochrome()) as demo:

    gr.Markdown('# 🎬 Text-to-Video AI Generator')
    gr.Markdown('Generate short videos from text using Latent Diffusion Models · ModelScope 1.7B')

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### ✍️ Prompt')
            prompt_in = gr.Textbox(
                label='Describe your video',
                placeholder='A cat playing in autumn leaves...',
                lines=3
            )
            gr.Markdown('### 🎨 Style')
            style_in  = gr.Dropdown(
                choices=list(STYLE_PRESETS.keys()),
                value='Cinematic',
                label='Style Preset',
                info='Automatically added to your prompt'
            )
            gr.Markdown('### 🎞️ Interpolation')
            interp_in = gr.Dropdown(
                choices=['None', '2x Smoother', '3x Smoother'],
                value='2x Smoother',
                label='Frame Interpolation',
                info='Adds frames for smoother motion'
            )
            gr.Markdown('### ⚙️ Settings')
            cfg_in    = gr.Slider(1, 15, value=7.5, step=0.5,
                                  label='CFG Scale',
                                  info='Low = creative | High = strict')
            steps_in  = gr.Slider(10, 50, value=25, step=5,
                                  label='Inference Steps',
                                  info='More = better quality, slower')
            seed_in   = gr.Number(value=42,
                                  label='Seed (same seed = same output)')
            btn       = gr.Button('🎬 Generate Video', variant='primary', size='lg')

            gr.Markdown('### 💡 Examples')
            gr.Examples(
                examples=[
                    ['Ocean waves on a beach at sunset'],
                    ['A hummingbird near a red flower'],
                    ['Clouds moving across the sky'],
                    ['A campfire in a forest at night'],
                    ['Snow falling in a quiet forest'],
                ],
                inputs=prompt_in
            )

        with gr.Column(scale=1):
            gr.Markdown('### 🎥 Output')
            video_out   = gr.Video(label='Generated Video')
            status_out  = gr.Textbox(
                label='Status',
                interactive=False,
                lines=3
            )
            gr.Markdown('### 📋 History')
            history_out = gr.Textbox(
                label='Past Generations (last 10)',
                interactive=False,
                lines=12,
                value='No generations yet.'
            )

    btn.click(
        fn=generate_video,
        inputs=[prompt_in, style_in, cfg_in, steps_in, seed_in, interp_in],
        outputs=[video_out, status_out, history_out]
    )

demo.launch(server_name="0.0.0.0", server_port=7860)

/tmp/ipykernel_796/4160146178.py:237: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Text-to-Video AI', theme=gr.themes.Monochrome()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6303ea7235172fae37.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import traceback
try:
    path, status, history = generate_video(
        'Snow falling in a quiet forest',
        'Cinematic',
        7.5,
        25,
        42,
        '2x Smoother'
    )
    print(status)
except Exception as e:
    traceback.print_ex

Loading model...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


✅ Model ready!
Prompt : Snow falling in a quiet forest, cinematic lighting, professional camera, smooth motion, high quality
Style  : Cinematic
CFG    : 7.5 | Steps: 25 | Seed: 42


  0%|          | 0/25 [00:00<?, ?it/s]

Frames : 16 → 31
✅ Done in 21.9s
Frames: 31 | FPS: 16 | Style: Cinematic
